# 01_RAG 检索实验 Notebook

这份 notebook 做三件事：

1. 验证 `01_RAG` 的导入、`.env` 和 embedding 配置是否正常
2. 复现 `原始文档 -> Document -> chunk` 的过程，并观察 overlap
3. 同时演示两套检索逻辑：
   - 当前 App 真实使用的 `Chroma 语义检索 + 相似度阈值过滤`
   - 教学用途的 `Hybrid 检索（Dense + BM25 + 融合）`

说明：本 notebook 只修改实验方式，不修改 `01_RAG` 源码。


In [1]:
import sys
import importlib
from pathlib import Path

from dotenv import load_dotenv

PROJECT = Path("01_RAG").resolve()
if not PROJECT.exists():
    raise FileNotFoundError(f"Project directory not found: {PROJECT}")

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

load_dotenv(PROJECT / ".env", override=True)

import config
import rag.embedder as embedder
import rag.vectorstore as vectorstore

importlib.reload(config)
importlib.reload(embedder)
importlib.reload(vectorstore)

print("project:", PROJECT)
print("config file:", config.__file__)
print("provider:", config.LLMConfig.provider())

embeddings = embedder.get_embeddings()
print("embeddings:", type(embeddings).__name__)


project: /Users/steven/LzmWorkSpace/Agent-Claude/LangGraph-Claude/01_RAG
config file: /Users/steven/LzmWorkSpace/Agent-Claude/LangGraph-Claude/01_RAG/config.py
provider: dashscope
embeddings: DashScopeEmbeddings


## 文档加载与分块

`load_pdf()` 先把 PDF 解析成按页组织的 `Document` 列表，再由 `chunk_documents()` 把较长页面切成多个 chunk。
overlap 只会发生在 **同一个 `Document` 被继续切开** 的情况下，不会跨页自动重叠。


In [2]:
from rag.loader import load_pdf
from rag.chunker import chunk_documents

sample_files = {
    "AI_Agent_技术白皮书.pdf": PROJECT / "data/documents/AI_Agent_技术白皮书.pdf",
    "刘泽明简历.pdf": PROJECT / "data/documents/刘泽明简历.pdf",
}

loaded_docs = {}
chunk_sets = {}

for name, path in sample_files.items():
    docs = load_pdf(str(path))
    chunks = chunk_documents(docs, chunk_size=500, chunk_overlap=50)
    loaded_docs[name] = docs
    chunk_sets[name] = chunks

    print(f"=== {name} ===")
    print(f"页数：{len(docs)} -> 块数：{len(chunks)}")
    print(f"第一块长度：{len(chunks[0].page_content)}")
    print(f"第一块元数据：{chunks[0].metadata}")
    print()


=== AI_Agent_技术白皮书.pdf ===
页数：7 -> 块数：12
第一块长度：71
第一块元数据：{'source': 'AI_Agent_技术白皮书.pdf', 'file_path': '/Users/steven/LzmWorkSpace/Agent-Claude/LangGraph-Claude/01_RAG/data/documents/AI_Agent_技术白皮书.pdf', 'page': 1, 'total_pages': 7, 'doc_id': '42dd2d5d8bdd', 'chunk_index': 0}

=== 刘泽明简历.pdf ===
页数：2 -> 块数：15
第一块长度：245
第一块元数据：{'source': '刘泽明简历.pdf', 'file_path': '/Users/steven/LzmWorkSpace/Agent-Claude/LangGraph-Claude/01_RAG/data/documents/刘泽明简历.pdf', 'page': 1, 'total_pages': 3, 'doc_id': '34816d19c54b', 'chunk_index': 0}



In [3]:
def shared_prefix_suffix(left: str, right: str, max_chars: int = 120) -> str:
    max_len = min(len(left), len(right), max_chars)
    for size in range(max_len, 0, -1):
        if left[-size:] == right[:size]:
            return left[-size:]
    return ""


def find_visible_overlap(chunks):
    for idx in range(len(chunks) - 1):
        left = chunks[idx]
        right = chunks[idx + 1]
        overlap = shared_prefix_suffix(left.page_content, right.page_content)
        if (
            left.metadata.get("source") == right.metadata.get("source")
            and left.metadata.get("page") == right.metadata.get("page")
            and overlap
        ):
            return idx, left, right, overlap
    return None


for name, chunks in chunk_sets.items():
    print(f"=== {name} ===")
    result = find_visible_overlap(chunks)
    if result is None:
        print("没有找到同页且可见重叠的相邻 chunk。")
    else:
        idx, left, right, overlap = result
        preview = min(len(overlap) + 40, 120)
        print(
            f"选中的相邻块: chunk[{idx}] -> chunk[{idx + 1}] | "
            f"page {left.metadata['page']} | overlap 长度: {len(overlap)}"
        )
        print("左块末尾：", repr(left.page_content[-preview:]))
        print("右块开头：", repr(right.page_content[:preview]))
    print()


=== AI_Agent_技术白皮书.pdf ===
选中的相邻块: chunk[1] -> chunk[2] | page 2 | overlap 长度: 41
左块末尾： '理需要多步骤推理的复杂任务，如：调研报告生成、代码调试、数据分析等。Agent\n的行动空间由其配置的工具集决定，可以无限扩展。\n1.3 Agent 的主要架构类型'
右块开头： '的行动空间由其配置的工具集决定，可以无限扩展。\n1.3 Agent 的主要架构类型\n目前业界主流的 Agent 架构类型分为以下四种：\n• ReAct（Reaso'

=== 刘泽明简历.pdf ===
选中的相邻块: chunk[2] -> chunk[3] | page 1 | overlap 长度: 47
左块末尾： '断提升MCP调用精度。\n●\n使用自定义拦截器实现敏感词检查、提示词规范等操作。\n●\n使用RAG检索与知识构建，将专业资料、设计规范、标书模板等存入向量数据库实现精准问答。\n●'
右块开头： '●\n使用RAG检索与知识构建，将专业资料、设计规范、标书模板等存入向量数据库实现精准问答。\n●\n引入ReAct+Reflection+Memory机制对上下文记忆持久化、对模'



## 向量库检查

下面先看当前持久化 Chroma 里已经有哪些文档。这个 notebook 默认 **复用现有 `01_RAG/data/vectorstore`**，并假定文档已提前入库。
如果缺少目标 PDF，请先在 App 中上传并完成索引。


In [4]:
from rag.vectorstore import get_vectorstore, list_documents, get_collection_stats

vs = get_vectorstore()
stats = get_collection_stats(vs)
indexed_docs = list_documents(vs)

print(stats)
print()

for doc in indexed_docs:
    print(
        f"- {doc['source']} | doc_id={doc['doc_id']} | "
        f"pages={doc['pages']} | chunks={doc['total_chunks']}"
    )

expected = {"AI_Agent_技术白皮书.pdf", "刘泽明简历.pdf"}
actual = {doc["source"] for doc in indexed_docs}
missing = expected - actual

if missing:
    print()
    print("缺少以下演示文档，请先在 App 中完成索引：", sorted(missing))


{'total_chunks': 37, 'collection_name': 'rag_knowledge_base', 'persist_dir': '/Users/steven/LzmWorkSpace/Agent-Claude/LangGraph-Claude/01_RAG/data/vectorstore'}

- AI_Agent_技术白皮书.pdf | doc_id=42dd2d5d8bdd | pages=[1, 2, 3, 4, 5, 6, 7] | chunks=12
- LangGraph_开发手册.pdf | doc_id=27449c8da210 | pages=[1, 2, 3, 4, 5] | chunks=10
- 刘泽明简历.pdf | doc_id=34816d19c54b | pages=[1, 2] | chunks=15


## 当前 App 的真实检索逻辑

当前主程序没有接入 `rag/retriever.py` 的 hybrid 路线，而是直接调用
`similarity_search_with_threshold()`，即：

`query -> Chroma 相似度检索 -> 阈值过滤`

注意：完整 App 在这一步之前还会先做 query rewrite；下面这段代码只演示 **检索阶段**。


In [7]:
from rag.vectorstore import similarity_search_with_threshold


def show_docs(docs, snippet_chars: int = 180):
    if not docs:
        print("没有检索到结果。")
        return

    for idx, doc in enumerate(docs, 1):
        meta = doc.metadata
        score = meta.get("similarity_score", "N/A")
        snippet = doc.page_content[:snippet_chars].replace("\n", " ")
        print(
            f"[{idx}] {meta.get('source')} | page={meta.get('page')} | "
            f"score={score}"
        )
        print(snippet)
        print("-" * 100)


def run_app_retrieval(query: str, k: int = 4, threshold: float = 0.3):
    docs = similarity_search_with_threshold(
        query=query,
        k=k,
        threshold=threshold,
        vectorstore=vs,
    )
    print(f"query: {query}")
    print(f"retrieved: {len(docs)}")
    show_docs(docs)
    return docs


In [6]:
app_queries = [
    "什么是 RAG？",
    "刘泽明最近在哪家公司工作？",
    "AI Agent 的主要架构类型有哪些？",
]

for query in app_queries:
    print("=" * 100)
    run_app_retrieval(query, k=4, threshold=0.3)
    print()


query: 什么是 RAG？
retrieved: 4
[1] AI_Agent_技术白皮书.pdf | page=3 | score=0.6404
第二章 RAG 技术详解 2.1 RAG 的定义与原理 RAG（Retrieval-Augmented Generation，检索增强生成）是一种将信息检索与文本生成相结合的技术框架。其核心思想是：在 LLM 生成回答之前，先从外部知识库中检索相关文档，将检索结果作为上下文注入 Prompt，从而使 LLM 能够基于最新、准确的私有知识回答问题，有效解决 L
----------------------------------------------------------------------------------------------------
[2] AI_Agent_技术白皮书.pdf | page=6 | score=0.6277
Token，无限用户，私有部署选项。 4.3 安全与合规 企业级 RAG 系统需满足以下安全合规要求： • 数据隔离：多租户场景下，各用户知识库完全隔离，向量索引按租户分区。 • 传输加密：所有 API 通信使用 TLS 1.3 加密。 • 访问控制：基于 RBAC 的权限管理，文档级别的访问权限控制。 • 审计日志：记录所有查询请求、文档操作，日志保留90
----------------------------------------------------------------------------------------------------
[3] AI_Agent_技术白皮书.pdf | page=6 | score=0.5841
第四章 产品规格与性能基准 4.1 RAG 系统性能指标 以下为企业级 RAG 系统的生产环境性能基准（基于 claude-sonnet -4-6 + Chroma + 混合检索）： 指标 目标值 说明 单次问答延迟 ≤ 5 秒 P90 延迟，含检索+生成 文档索引速度 ≤ 30 秒/100页 PDF解析+分块+向量化 检索准确率（MRR@4） ≥ 0.75
-------------------------------------------------------------------------------------

## Hybrid 检索实验版

这里不直接调用仓库里的 `rag.retriever.build_hybrid_retriever()`，因为当前版本那条路径会把
`score_threshold` 传进 `search_type="similarity"`，在现有依赖下会报：

`TypeError: query() got an unexpected keyword argument 'score_threshold'`

所以下面在 notebook 内单独构建一个**可运行的教学版 hybrid retriever**：

- Dense: `Chroma.as_retriever(search_type="similarity")`
- Sparse: `BM25Retriever`
- Fusion: `EnsembleRetriever`


In [8]:
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

raw = vs.get(include=["documents", "metadatas"])
all_chunks = [
    Document(page_content=doc, metadata=meta or {})
    for doc, meta in zip(raw.get("documents", []), raw.get("metadatas", []))
]

if not all_chunks:
    raise ValueError("当前向量库为空，无法构建 hybrid retriever。")

semantic_retriever = vs.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 6},
)
bm25_retriever = BM25Retriever.from_documents(all_chunks)
bm25_retriever.k = 6

hybrid_retriever = EnsembleRetriever(
    retrievers=[semantic_retriever, bm25_retriever],
    weights=[0.6, 0.4],
)

print(f"Loaded {len(all_chunks)} chunks for hybrid retrieval.")


Loaded 37 chunks for hybrid retrieval.


In [9]:
from rag.retriever import retrieve_with_hybrid
def run_hybrid_retrieval(query: str, top_k: int = 4):
    # docs = hybrid_retriever.invoke(query)[:top_k]
    docs = retrieve_with_hybrid(query, top_k=top_k)
    print(f"query: {query}")
    print(f"retrieved: {len(docs)}")
    show_docs(docs)
    return docs


hybrid_queries = [
    "什么是 RAG？",
    "刘泽明最近在哪家公司工作？",
    "AI Agent 的主要架构类型有哪些？",
]

for query in hybrid_queries:
    print("=" * 100)
    run_hybrid_retrieval(query, top_k=4)
    print()


query: 什么是 RAG？
retrieved: 4
[1] AI_Agent_技术白皮书.pdf | page=6 | score=N/A
Token，无限用户，私有部署选项。 4.3 安全与合规 企业级 RAG 系统需满足以下安全合规要求： • 数据隔离：多租户场景下，各用户知识库完全隔离，向量索引按租户分区。 • 传输加密：所有 API 通信使用 TLS 1.3 加密。 • 访问控制：基于 RBAC 的权限管理，文档级别的访问权限控制。 • 审计日志：记录所有查询请求、文档操作，日志保留90
----------------------------------------------------------------------------------------------------
[2] AI_Agent_技术白皮书.pdf | page=3 | score=N/A
第二章 RAG 技术详解 2.1 RAG 的定义与原理 RAG（Retrieval-Augmented Generation，检索增强生成）是一种将信息检索与文本生成相结合的技术框架。其核心思想是：在 LLM 生成回答之前，先从外部知识库中检索相关文档，将检索结果作为上下文注入 Prompt，从而使 LLM 能够基于最新、准确的私有知识回答问题，有效解决 L
----------------------------------------------------------------------------------------------------
[3] AI_Agent_技术白皮书.pdf | page=6 | score=N/A
第四章 产品规格与性能基准 4.1 RAG 系统性能指标 以下为企业级 RAG 系统的生产环境性能基准（基于 claude-sonnet -4-6 + Chroma + 混合检索）： 指标 目标值 说明 单次问答延迟 ≤ 5 秒 P90 延迟，含检索+生成 文档索引速度 ≤ 30 秒/100页 PDF解析+分块+向量化 检索准确率（MRR@4） ≥ 0.75
----------------------------------------------------------------------------------------------

## 两套检索结果对比

下面把同一个 query 同时跑一遍，方便观察：

- 纯语义检索什么时候已经足够
- hybrid 在关键词命中上什么时候更稳定
- 为什么“文档里写了 hybrid”不等于“主程序真的在用 hybrid”


In [10]:
def compare_retrievals(query: str, k: int = 4, threshold: float = 0.3):
    print("#" * 100)
    print(f"Compare query: {query}")
    print("\n[App-style retrieval]")
    app_docs = run_app_retrieval(query, k=k, threshold=threshold)
    print("\n[Hybrid retrieval]")
    hybrid_docs = run_hybrid_retrieval(query, top_k=k)
    return {"app": app_docs, "hybrid": hybrid_docs}


comparison_queries = [
    "什么是 RAG？",
    "刘泽明最近在哪家公司工作？",
    "AI Agent 的主要架构类型有哪些？",
]

comparison_results = {}
for query in comparison_queries:
    comparison_results[query] = compare_retrievals(query)
    print()


####################################################################################################
Compare query: 什么是 RAG？

[App-style retrieval]
query: 什么是 RAG？
retrieved: 4
[1] AI_Agent_技术白皮书.pdf | page=3 | score=0.6404
第二章 RAG 技术详解 2.1 RAG 的定义与原理 RAG（Retrieval-Augmented Generation，检索增强生成）是一种将信息检索与文本生成相结合的技术框架。其核心思想是：在 LLM 生成回答之前，先从外部知识库中检索相关文档，将检索结果作为上下文注入 Prompt，从而使 LLM 能够基于最新、准确的私有知识回答问题，有效解决 L
----------------------------------------------------------------------------------------------------
[2] AI_Agent_技术白皮书.pdf | page=6 | score=0.6277
Token，无限用户，私有部署选项。 4.3 安全与合规 企业级 RAG 系统需满足以下安全合规要求： • 数据隔离：多租户场景下，各用户知识库完全隔离，向量索引按租户分区。 • 传输加密：所有 API 通信使用 TLS 1.3 加密。 • 访问控制：基于 RBAC 的权限管理，文档级别的访问权限控制。 • 审计日志：记录所有查询请求、文档操作，日志保留90
----------------------------------------------------------------------------------------------------
[3] AI_Agent_技术白皮书.pdf | page=6 | score=0.5841
第四章 产品规格与性能基准 4.1 RAG 系统性能指标 以下为企业级 RAG 系统的生产环境性能基准（基于 claude-sonnet -4-6 + Chroma + 混合检索）： 指标 目标值 说明 单次问答延迟 ≤ 5 秒 P90